# Predição v2 — MultiOutputRegressor

Este notebook concentra preparação, treinamento, avaliação e exportação da versão 2.
A arquitetura deriva de `../../MultiOutputRegressor.ipynb`: Random Forest, XGBoost e CatBoost em `StackingRegressor`, com `RidgeCV` e `MultiOutputRegressor` para os oito metais.
O `main.py` e o YOLO permanecem intactos. O modelo anterior está preservado em `historico/random_forest/`.

As sete entradas atuais, incluindo OD medido, são preservadas. A separação é por estação; o teste não participa da escolha dos parâmetros. A transformação quantílica dos alvos mantém a saída dentro das concentrações observadas no ajuste, evitando concentrações negativas do combinador linear.

Selecione o ambiente com `requirements-training.txt` e execute as células em ordem. A preparação só baixa/reconstrói quando o CSV normalizado ainda não existe. Cada treinamento gera uma pasta própria em `runs/`. A última célula publica modelo e manifesto nos caminhos que a API já utiliza.

In [ ]:
import os
from pathlib import Path
import csv
import hashlib
import importlib.metadata
import io
import json
import math
import shutil
from collections import Counter
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from zipfile import ZipFile
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

local = Path.cwd().resolve()
possible_roots = [local, *local.parents]
VERSION_ROOT = next((p for p in possible_roots if (p / "preservation_checks.json").is_file() and (p / "data/gemstat_prepared").is_dir()), None)
if VERSION_ROOT is None:
    for p in possible_roots:
        candidate = p / "back-end/ia/app/services/Predict/model/metals_v2_20261004"
        if candidate.is_dir():
            VERSION_ROOT = candidate
            break
if VERSION_ROOT is None:
    raise FileNotFoundError("Abra o notebook na pasta da versão ou na raiz do repositório.")
PREDICT_ROOT = VERSION_ROOT.parents[1]
APP_ROOT = PREDICT_ROOT.parents[1]
MAIN_PATH = APP_ROOT / "main.py"
MAIN_SHA256_BEFORE = hashlib.sha256(MAIN_PATH.read_bytes()).hexdigest()
VERSION = VERSION_ROOT.name
ARCHIVE = VERSION_ROOT / "data/external/gemstat/GFQA_v3.zip"
OUTPUT = VERSION_ROOT / "data/gemstat_prepared"
SOURCE_URL = "https://zenodo.org/records/18459694/files/GFQA_v3.zip?download=1"
SOURCE_MD5 = "00f3ea19ce529753977eb3aeb08fbc47"
SOURCE_DOI = "10.5281/zenodo.18459694"
SEED = 42
MAX_PER_STATION = 120
THREADS = 2
RUN_DIR = VERSION_ROOT / "runs" / datetime.now(timezone.utc).strftime("multioutput_stacking_%Y%m%dT%H%M%S%fZ")
print("Pasta da versão:", VERSION_ROOT)
print("Saída deste treinamento:", RUN_DIR)

Pasta da versão: C:\Users\lf446\OneDrive\Documentos\GitHub\WaterPath_TCC\back-end\ia\app\services\Predict\model\metals_v2_20261004
Saída deste treinamento: C:\Users\lf446\OneDrive\Documentos\GitHub\WaterPath_TCC\back-end\ia\app\services\Predict\model\metals_v2_20261004\runs\multioutput_stacking_20261004T164732579726Z


## Contrato de dados

As listas são definidas aqui; não há importação de scripts locais. A preparação preserva TN para auditoria, mas o modelo mantém as sete entradas do contrato atual, sem nitrogênio total. A retirada anterior foi avaliada com o regressor anterior; essa evidência não é atribuída ao novo stacking.

In [2]:
FIELDS = {
    "temperatura": ("T, °C", "°C"),
    "ph": ("рН", "pH"),
    "condutividade_eletrica": ("EC, µS/cm", "µS/cm"),
    "oxigenio_dissolvido": ("DO, mg/L", "mg/L"),
    "solidos_suspensos_totais": ("TSS, mg/L", "mg/L"),
    "carbono_organico_total": ("TOC, mg/L", "mg/L"),
    "nitrogenio_total": ("TN, mg/L", "mg/L"),
    "fosforo_total": ("TP, µg/L", "µg P/L"),
}
FEATURES = [column for column, _ in FIELDS.values()]
TARGETS = ["Fe, mg/L", "Mn, mg/L", "Cr, µg/L", "Ni, µg/L",
           "Cu, µg/L", "Zn, µg/L", "Cd, µg/L", "Pb, µg/L"]
REQUIRED_FIELDS = ("temperatura", "ph", "condutividade_eletrica", "oxigenio_dissolvido")

feature_columns = ['T, °C', 'рН', 'EC, µS/cm', 'DO, mg/L', 'TSS, mg/L', 'TOC, mg/L', 'TP, µg/L']
result = list(TARGETS)
selected_fields = [field for field, (column, _) in FIELDS.items() if column in feature_columns]
display(pd.DataFrame([{"campo":field,"coluna":column,"unidade":unit,"obrigatorio":field in REQUIRED_FIELDS} for field,(column,unit) in FIELDS.items() if column in feature_columns]))

,campo,coluna,unidade,obrigatorio
0,temperatura,"T, °C",°C,True
1,ph,рН,pH,True
2,condutividade_eletrica,"EC, µS/cm",µS/cm,True
3,oxigenio_dissolvido,"DO, mg/L",mg/L,True
4,solidos_suspensos_totais,"TSS, mg/L",mg/L,False
5,carbono_organico_total,"TOC, mg/L",mg/L,False
6,fosforo_total,"TP, µg/L",µg P/L,False


## Preparação e aquisição, dentro do notebook

A junção usa estação/data/hora/profundidade exatas e apenas os oito metais totais. Unidades incompatíveis, conflitos, resultados integrados e baixa qualidade são excluídos. `<LOD` usa LOD/2. OD acima de 40 mg/L é excluído por triagem de plausibilidade, sem adivinhar porcentagens. O ZIP conserva a fonte pública e é validado pelo hash publicado.

In [3]:
METAL_CODES = {
    "Fe-Tot": ("Iron.csv", TARGETS[0], "mg/l"),
    "Mn-Tot": ("Manganese.csv", TARGETS[1], "mg/l"),
    "Cr-Tot": ("Chromium.csv", TARGETS[2], "ug/l"),
    "Ni-Tot": ("Nickel.csv", TARGETS[3], "ug/l"),
    "Cu-Tot": ("Copper.csv", TARGETS[4], "ug/l"),
    "Zn-Tot": ("Zinc.csv", TARGETS[5], "ug/l"),
    "Cd-Tot": ("Cadmium.csv", TARGETS[6], "ug/l"),
    "Pb-Tot": ("Lead.csv", TARGETS[7], "ug/l"),
}
INPUT_CODES = {
    "TEMP": ("Temperature.csv", FIELDS["temperatura"][0], "degc"),
    "pH": ("pH.csv", FIELDS["ph"][0], "ph"),
    "EC": ("Electrical_Conductance.csv", FIELDS["condutividade_eletrica"][0], "us/cm"),
    "O2-Dis": ("Dissolved_Gas.csv", FIELDS["oxigenio_dissolvido"][0], "mg/l"),
    "TSS": ("Water.csv", FIELDS["solidos_suspensos_totais"][0], "mg/l"),
    "TOC": ("Carbon.csv", FIELDS["carbono_organico_total"][0], "mg/l"),
    "TN": ("Other_Nitrogen.csv", FIELDS["nitrogenio_total"][0], "mg/l"),
    "TP": ("Phosphorus.csv", FIELDS["fosforo_total"][0], "ug/l"),
}


def digest(path, algorithm="sha256"):
    hasher = hashlib.new(algorithm)
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def download_archive(path=ARCHIVE):
    path = Path(path)
    if path.is_file() and digest(path, "md5") == SOURCE_MD5:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".download")
    with urlopen(Request(SOURCE_URL, headers={"User-Agent": "WaterPath-dataset/2.0"}), timeout=90) as response, temporary.open("wb") as handle:
        for block in iter(lambda: response.read(1024 * 1024), b""):
            handle.write(block)
    if digest(temporary, "md5") != SOURCE_MD5:
        raise ValueError("Hash do arquivo GEMStat diverge do publicado no Zenodo")
    temporary.replace(path)
    return path


def rows(archive, name):
    with archive.open(name) as handle:
        yield from csv.DictReader(io.TextIOWrapper(handle, encoding="cp1252", newline=""))


def sample_key(row):
    # Hora e profundidade fazem parte da chave; não junta apenas por dia/estação.
    return tuple(row[key].strip() for key in ("GEMS Station Number", "Sample Date", "Sample Time", "Depth"))


def convert_value(row, expected_unit):
    unit = row["Unit"].strip().lower().replace("µ", "u").replace("μ", "u").replace("°", "deg")
    unit = unit.replace(" ", "")
    if expected_unit == "degc" and unit in {"c", "degc", "degreec", "degrees_c"}:
        multiplier = 1
    elif expected_unit == "ph" and unit in {"ph", "phunits", "", "1", "dimensionless", "---"}:
        multiplier = 1
    elif unit == expected_unit:
        multiplier = 1
    elif (unit, expected_unit) == ("ug/l", "mg/l"):
        multiplier = 0.001
    elif (unit, expected_unit) == ("mg/l", "ug/l"):
        multiplier = 1000
    elif (unit, expected_unit) == ("ms/m", "us/cm"):
        multiplier = 10
    else:
        raise ValueError(f"Unidade não compatível: {unit!r} -> {expected_unit}")
    value = float(row["Value"]) * multiplier
    flag = row["Value Flags"].strip()
    if flag not in {"", "<"} or not math.isfinite(value) or (value < 0 and expected_unit != "degc"):
        raise ValueError("Resultado inválido/estimado ou acima do limite")
    return value / 2 if flag == "<" else value


def prepare(archive_path=ARCHIVE, output=OUTPUT):
    output = Path(output)
    if output.exists():
        raise FileExistsError(f"Preserve a versão existente e escolha outra saída: {output}")
    if digest(archive_path, "md5") != SOURCE_MD5:
        raise ValueError("Integridade GEMStat não confirmada")
    audit = {}
    with ZipFile(archive_path) as archive:
        stations = {r["GEMS Station Number"]: r for r in rows(archive, "GEMStat_station_metadata.csv")}
        allowed = {key for key, r in stations.items() if r["Water Type"] in {"River station", "Lake station", "Reservoir station"}}
        samples = {}
        # Cadmium primeiro limita a memória; cada metal reduz a interseção.
        order = ["Cd-Tot", "Pb-Tot", "Cr-Tot", "Ni-Tot", "Cu-Tot", "Zn-Tot", "Fe-Tot", "Mn-Tot"]
        for code in order + list(INPUT_CODES):
            filename, column, unit = (METAL_CODES | INPUT_CODES)[code]
            matched, conflicts, counters = {}, set(), Counter()
            for row in rows(archive, filename):
                if row["Parameter Code"] != code or row["GEMS Station Number"] not in allowed:
                    continue
                key = sample_key(row)
                if samples and key not in samples:
                    continue
                if code != order[0] and not samples:
                    raise ValueError("Interseção de amostras vazia")
                if row["Data Quality"] not in {"Good", "Fair", "Unknown"} or row["Integrated Value"].strip():
                    counters["quality_or_integrated_excluded"] += 1
                    continue
                try:
                    value = convert_value(row, unit)
                except (ValueError, TypeError):
                    counters["invalid_value_or_unit"] += 1
                    continue
                if key in matched and not math.isclose(matched[key][0], value, rel_tol=1e-9, abs_tol=1e-12):
                    conflicts.add(key)
                matched[key] = (value, row["Value Flags"].strip() == "<")
            for key in conflicts:
                matched.pop(key, None)
            if code == order[0]:
                samples = {key: {column: value, "censored_targets": int(censored)} for key, (value, censored) in matched.items()}
            else:
                for key, record in list(samples.items()):
                    if key in matched:
                        record[column] = matched[key][0]
                        if code in METAL_CODES:
                            record["censored_targets"] += int(matched[key][1])
                    elif code in METAL_CODES:
                        del samples[key]
            audit[code] = {**counters, "conflicting_samples": len(conflicts), "matched": len(matched), "remaining": len(samples)}
            print(f"{code}: {len(matched)} medições; {len(samples)} amostras na interseção", flush=True)
        required = [FIELDS[field][0] for field in REQUIRED_FIELDS]
        records = []
        quality_screen = Counter()
        for key, values in sorted(samples.items()):
            if not all(column in values for column in required):
                continue
            if not 0 <= values[FIELDS["ph"][0]] <= 14:
                quality_screen["ph_outside_0_14"] += 1
                continue
            # Triagem conservadora de possíveis erros de unidade/transcrição.
            # Não converte saturação (%) em mg/L nem corrige valores por adivinhação.
            if values[FIELDS["oxigenio_dissolvido"][0]] > 40:
                quality_screen["oxygen_above_40_mg_l"] += 1
                continue
            station, date, time, depth = key
            records.append({"sample_id": "|".join(key), "station": station, "date": date,
                            "time": time, "sample_depth_m": depth, "country": stations[station]["Country Name"],
                            "source": SOURCE_DOI, "target_fraction": "total", **values})
        if len(records) < 100:
            raise ValueError(f"Apenas {len(records)} amostras válidas; revise a cobertura")
        output.mkdir(parents=True)
        columns = ["sample_id", "station", "date", "time", "sample_depth_m", "country", "source", "target_fraction", "censored_targets"] + FEATURES + TARGETS
        with (output / "samples.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns)
            writer.writeheader()
            writer.writerows(records)
        for name in ("GEMStat_parameter_metadata.csv", "GEMStat_methods_metadata.csv", "README_output_format.txt"):
            (output / name).write_bytes(archive.read(name))
    write_json(output / "manifest.json", {
        "version": output.name, "source_doi": SOURCE_DOI, "source_url": SOURCE_URL, "license": "CC BY 4.0",
        "attribution": "Heinle, Moritz; Lisniak, Dmytro; Saile, Philipp; UNEP GEMS/Water Data Centre, GFQA v3 (2026)",
        "archive_sha256": digest(archive_path), "archive_md5": SOURCE_MD5,
        "samples_sha256": digest(output / "samples.csv"), "records": len(records),
        "stations": len({r["station"] for r in records}), "countries": dict(Counter(r["country"] for r in records)),
        "feature_observations": {column: sum(column in r for r in records) for column in FEATURES},
        "required_measured_features": required, "features": FEATURES, "targets": TARGETS,
        "join": "exact station + date + time + depth; total fraction only; conflicting duplicates excluded",
        "censored_values": "<LOD -> LOD/2, approximate labels retained with count; no fabricated targets",
        "quality": "Good/Fair/Unknown; excludes estimated/suspect/contamination/pending and integrated observations",
        "audit": audit,
        "quality_screen": {"excluded": dict(quality_screen), "oxygen_max_mg_l": 40,
                           "reason": "Engineering plausibility screen, not a legal limit or universal physical maximum; suspicious values are excluded, never converted by guessing."},
        "limitations": ["Default times may not uniquely identify laboratory aliquots.", "Observations below quantification limit are approximate.", "No paired images; this dataset does not train YOLO.", "No Brazilian stations in this archive; validate locally before claiming geographic generalization."]})
    print(f"Dataset preparado: {len(records)} amostras em {output}", flush=True)
    return output

In [4]:
# Conferências de unidades; estas funções são implementadas na célula acima.
row = {"Unit": "µg/l", "Value": "1000", "Value Flags": ""}
assert convert_value(row, "mg/l") == 1
assert convert_value({**row, "Value Flags": "<"}, "mg/l") == 0.5
try:
    convert_value({**row, "Unit": "%"}, "mg/l")
except ValueError:
    pass
else:
    raise AssertionError("Saturação em % não pode virar OD em mg/L.")

if not (OUTPUT / "samples.csv").is_file():
    download_archive(ARCHIVE)
    prepare(ARCHIVE, OUTPUT)
dataset_manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
assert digest(OUTPUT / "samples.csv") == dataset_manifest["samples_sha256"]
original = pd.read_csv(OUTPUT / "samples.csv")
assert original.sample_id.is_unique
assert original[result].notna().all().all()
assert original[[FIELDS[field][0] for field in REQUIRED_FIELDS]].notna().all().all()
assert np.isfinite(original[result].to_numpy()).all()
assert (original[result] >= 0).all().all()
print(f"{len(original):,} amostras; {original.station.nunique()} estações; {original.country.nunique()} países")
display(original.groupby("country").agg(amostras=("sample_id", "size"), estacoes=("station", "nunique")))

14,034 amostras; 777 estações; 9 países


,amostras,estacoes
country,,
Austria,1593,94
Canada,3077,88
Finland,29,4
Ireland,1539,73
Italy,1930,233
Mexico,135,133
Serbia,3579,41
Sweden,2151,110
United States of America (the),1,1


## Separação por estação

O limite de 120 observações por estação mantém datas distribuídas no histórico. Os mesmos grupos de treino, validação e teste da primeira v2 são conservados para comparação. O stacking também constrói suas previsões fora da amostra com três folds separados por estação. A imputação está em cada estimador base, portanto é ajustada novamente em cada fold.

In [5]:
frame = pd.concat([
    group.sort_values(["date", "sample_id"]).iloc[np.unique(np.linspace(0, len(group) - 1, min(MAX_PER_STATION, len(group))).astype(int))]
    for _, group in original.groupby("station", sort=True)
], ignore_index=True)
train_val, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(frame, groups=frame.station))
train_rel, val_rel = next(GroupShuffleSplit(n_splits=1, test_size=0.18, random_state=43).split(frame.iloc[train_val], groups=frame.iloc[train_val].station))
partitions = {"train": train_val[train_rel], "val": train_val[val_rel], "test": test_idx}
station_sets = {name: set(frame.iloc[idx].station) for name, idx in partitions.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert station_sets[a].isdisjoint(station_sets[b])
    assert set(frame.iloc[partitions[a]].sample_id).isdisjoint(frame.iloc[partitions[b]].sample_id)
X, Y = frame[feature_columns], frame[result]
tr, va, te = [partitions[name] for name in ("train", "val", "test")]
TrainX, TrainY = X.iloc[tr], Y.iloc[tr]
ValX, ValY = X.iloc[va], Y.iloc[va]
TestX, TestY = X.iloc[te], Y.iloc[te]
normalization_scale = np.maximum(TrainY.std(ddof=0).to_numpy(), 1e-9)
display(pd.DataFrame([{"particao":name,"amostras":len(idx),"estacoes":len(station_sets[name])} for name,idx in partitions.items()]))

,particao,amostras,estacoes
0,train,9361,541
1,val,1955,119
2,test,1662,117


## MultiOutputRegressor + StackingRegressor

Mantém-se a estrutura do notebook de referência: RF/XGBoost/CatBoost, combinados por RidgeCV, com um regressor por metal. Duas configurações regularizadas são comparadas na validação; não se repete a busca extensa do notebook antigo no teste. A transformação quantílica dos alvos é ajustada apenas na partição de treinamento de cada candidato; não utiliza validação nem teste.

Cada base inclui seu próprio `SimpleImputer`. O `MultiOutputRegressor` trabalha sequencialmente para limitar memória; os estimadores usam dois threads no treino e um na API.

In [6]:
def make_model(groups, config):
    folds = list(GroupKFold(n_splits=3).split(np.zeros((len(groups), 1)), groups=groups))
    rf = RandomForestRegressor(n_estimators=100, max_depth=config["rf_depth"], min_samples_leaf=5, random_state=SEED, n_jobs=THREADS)
    xgboost = XGBRegressor(n_estimators=100, max_depth=config["xgb_depth"], learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=SEED, n_jobs=THREADS)
    catboost = CatBoostRegressor(iterations=100, depth=config["cat_depth"], learning_rate=0.05, l2_leaf_reg=10, loss_function="RMSE", random_seed=SEED, verbose=False, thread_count=THREADS, allow_writing_files=False)
    estimators = [(name, Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", estimator)]))
                  for name, estimator in (("rf", rf), ("xgboost", xgboost), ("catboost", catboost))]
    stackingModel = StackingRegressor(estimators=estimators, final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 13)), cv=folds, n_jobs=1)
    transformed = TransformedTargetRegressor(regressor=stackingModel, transformer=QuantileTransformer(n_quantiles=min(200, len(groups)), output_distribution="normal", random_state=SEED))
    return MultiOutputRegressor(transformed, n_jobs=1)


def normalized_mae(y, prediction):
    return float(np.mean(np.mean(np.abs(np.asarray(y) - prediction), axis=0) / normalization_scale))


def metric_report(y, prediction):
    return {"normalized_mae": normalized_mae(y, prediction), "per_target": {
        name: {"mae": float(mean_absolute_error(y.iloc[:,i], prediction[:,i])),
               "rmse": float(mean_squared_error(y.iloc[:,i], prediction[:,i]) ** 0.5),
               "r2": float(r2_score(y.iloc[:,i], prediction[:,i])), "unit": name.split(", ",1)[1]}
        for i,name in enumerate(result)}}

configs = {
    "stacking_regularized": {"rf_depth": 5, "xgb_depth": 3, "cat_depth": 4},
    "stacking_reference": {"rf_depth": 8, "xgb_depth": 5, "cat_depth": 4},
}
validation_candidates = {}
for name, config in configs.items():
    print(f"Treinando {name}: oito alvos, três bases e CV por estação...", flush=True)
    multiModels = make_model(frame.iloc[tr].station.to_numpy(), config)
    multiModels.fit(TrainX, TrainY)
    prediction = multiModels.predict(ValX)
    assert prediction.shape == (len(ValX), len(result))
    assert np.isfinite(prediction).all() and (prediction >= 0).all()
    validation_candidates[name] = metric_report(ValY, prediction)
    print(f"{name}: erro normalizado validação {validation_candidates[name]['normalized_mae']:.6f}", flush=True)
    del multiModels
winner = min(validation_candidates, key=lambda name: validation_candidates[name]["normalized_mae"])
baseline = DummyRegressor(strategy="median").fit(TrainX, TrainY)
validation_baseline = metric_report(ValY, baseline.predict(ValX))
print("Configuração escolhida somente pela validação:", winner)
display(pd.DataFrame([{"modelo":name,"erro_normalizado":metric["normalized_mae"]} for name,metric in validation_candidates.items()] + [{"modelo":"mediana","erro_normalizado":validation_baseline["normalized_mae"]}]))

Treinando stacking_regularized: oito alvos, três bases e CV por estação...
stacking_regularized: erro normalizado validação 0.157086
Treinando stacking_reference: oito alvos, três bases e CV por estação...
stacking_reference: erro normalizado validação 0.154090
Configuração escolhida somente pela validação: stacking_reference


,modelo,erro_normalizado
0,stacking_regularized,0.157086
1,stacking_reference,0.154090
2,mediana,0.196373


## Ajuste final e avaliação

O modelo escolhido é reajustado em treino + validação; o teste continua fora do ajuste. A comparação com o Random Forest anterior usa exatamente o mesmo teste da v2 e é exploratória, pois esse conjunto já foi avaliado anteriormente. O resultado não é usado para escolher outro candidato.

In [7]:
print("Ajuste final em treino + validação...", flush=True)
best_model = make_model(frame.iloc[train_val].station.to_numpy(), configs[winner])
best_model.fit(X.iloc[train_val], Y.iloc[train_val])
assert list(best_model.feature_names_in_) == feature_columns
prediction = best_model.predict(TestX)
assert np.isfinite(prediction).all() and (prediction >= 0).all()
test_metrics = metric_report(TestY, prediction)
baseline.fit(X.iloc[train_val], Y.iloc[train_val])
test_baseline = metric_report(TestY, baseline.predict(TestX))
previous_path = VERSION_ROOT / "historico/random_forest/quality_model.joblib"
previous_metrics = None
if previous_path.is_file():
    previous = joblib.load(previous_path)
    previous_metrics = metric_report(TestY, previous.predict(TestX[list(previous.feature_names_in_)]))
    del previous
print("Erro normalizado stacking:", test_metrics["normalized_mae"])
print("Erro normalizado mediana:", test_baseline["normalized_mae"])
print("Erro normalizado Random Forest anterior:", previous_metrics["normalized_mae"] if previous_metrics else "indisponível")
display(pd.DataFrame(test_metrics["per_target"]).T)
print("Redução de erro contra a mediana (%):", 100 * (1 - test_metrics["normalized_mae"] / test_baseline["normalized_mae"]))

Ajuste final em treino + validação...
Erro normalizado stacking: 0.13047115133005965
Erro normalizado mediana: 0.17262513501012103
Erro normalizado Random Forest anterior: 0.1381771789595458


,mae,rmse,r2,unit
"Fe, mg/L",0.359441,1.913831,0.315098,mg/L
"Mn, mg/L",0.022892,0.056271,0.405003,mg/L
"Cr, µg/L",0.710687,2.54489,0.362593,µg/L
"Ni, µg/L",2.968338,18.703176,0.00812,µg/L
"Cu, µg/L",2.861397,16.264386,0.035204,µg/L
"Zn, µg/L",12.617216,67.865335,0.029694,µg/L
"Cd, µg/L",0.042954,0.310275,0.101281,µg/L
"Pb, µg/L",0.630384,3.842265,0.059923,µg/L


Redução de erro contra a mediana (%): 24.41937767497756


## Exportação, manifesto e verificação

O artefato contém somente classes das bibliotecas instaladas. Nenhuma função ou classe definida neste notebook é necessária ao carregar o joblib em outro processo. O modelo anterior fica em `historico/` e cada execução preserva seus próprios artefatos em `runs/`.

In [8]:
RUN_DIR.mkdir(parents=True, exist_ok=False)
limits = {key:1 for key in best_model.get_params(deep=True) if key.split("__")[-1] in {"n_jobs", "thread_count", "nthread"}}
best_model.set_params(**limits)
model_path = RUN_DIR / "quality_model.joblib"
joblib.dump(best_model, model_path, compress=3)
loaded = joblib.load(model_path)
assert isinstance(loaded, MultiOutputRegressor)
assert all(isinstance(est.regressor_, StackingRegressor) for est in loaded.estimators_)
np.testing.assert_allclose(loaded.predict(TestX.iloc[:5]), best_model.predict(TestX.iloc[:5]))

manifest = {
    "version": VERSION, "format": "joblib", "model_file": model_path.name,
    "model_sha256": digest(model_path), "dataset_manifest": dataset_manifest,
    "sklearn_version": importlib.metadata.version("scikit-learn"),
    "packages": {name:importlib.metadata.version(name) for name in ("scikit-learn","joblib","numpy","pandas","scipy","xgboost-cpu","catboost")},
    "estimator": "MultiOutputRegressor(TransformedTargetRegressor(StackingRegressor[RF,XGBoost,CatBoost; RidgeCV], QuantileTransformer))",
    "reference_notebook": "../../MultiOutputRegressor.ipynb",
    "reference_notebook_sha256": digest(PREDICT_ROOT / "MultiOutputRegressor.ipynb"),
    "training_notebook": "MultiOutputRegressor_v2.ipynb",
    "selected_configuration": winner, "hyperparameters": configs[winner],
    "features": feature_columns, "fields": selected_fields, "required_fields": list(REQUIRED_FIELDS),
    "targets": result, "target_fraction": "total",
    "removed_low_contribution_features": [FIELDS["nitrogenio_total"][0]],
    "feature_removal_basis": "Seven-input API contract retained; nitrogen removal was evaluated with the previous RF, not this stacking model.",
    "selection": "validation normalized MAE; group-separated stacking folds; two fixed configurations",
    "validation_candidates": validation_candidates, "validation_baseline": validation_baseline,
    "test": test_metrics, "test_baseline": test_baseline, "previous_model_test": previous_metrics,
    "test_better_than_median": test_metrics["normalized_mae"] < test_baseline["normalized_mae"],
    "target_normalization_scale": dict(zip(result,map(float,normalization_scale))),
    "original_records": len(original), "training_pool_records": len(frame), "max_per_station": MAX_PER_STATION,
    "split": {name:{"rows":len(idx),"stations":sorted(station_sets[name])} for name,idx in partitions.items()},
    "fit_partitions": ["train","val"], "test_used_for_selection": False,
    "stacking_cv": {"method":"GroupKFold","folds":3,"group":"station","imputation":"inside each base-estimator pipeline"},
    "feature_ranges": {column:{"min":float(X.iloc[train_val][column].min()),"max":float(X.iloc[train_val][column].max()),"median":float(X.iloc[train_val][column].median())} for column in feature_columns},
    "main_sha256_before_training": MAIN_SHA256_BEFORE,
    "limitations": ["Estimates current sample concentrations, not future dates.", "LOD/2 labels are approximations.", "No Brazilian stations; local validation remains necessary.", "Test stations are disjoint but this holdout was evaluated by the prior v2; comparisons are exploratory.", "Quantile target inversion is bounded by observed fitting concentrations; extremes outside that range cannot be extrapolated.", "Optional missing inputs use medians fitted inside each base estimator."]
}
write_json(RUN_DIR / "manifest.json", manifest)
for name, idx in partitions.items():
    frame.iloc[idx].to_csv(RUN_DIR / f"{name}_samples.csv", index=False)
first = frame.iloc[te[0]]
write_json(RUN_DIR / "example_request.json", {field:float(first[FIELDS[field][0]]) if pd.notna(first[FIELDS[field][0]]) else None for field in selected_fields})
assert digest(MAIN_PATH) == MAIN_SHA256_BEFORE
print("Joblib e manifesto verificados:", RUN_DIR)

Joblib e manifesto verificados: C:\Users\lf446\OneDrive\Documentos\GitHub\WaterPath_TCC\back-end\ia\app\services\Predict\model\metals_v2_20261004\runs\multioutput_stacking_20261004T164732579726Z


## Disponibilizar para a API sem alterar main.py

Esta célula é parte da execução solicitada: copia somente artefatos de inferência e de avaliação para a raiz da pasta da versão, no caminho que a API já carrega. O código da API e os arquivos do YOLO não são escritos. Para uma execução futura apenas experimental, pare antes desta célula. Reinicie uma API que já esteja em execução para limpar seu cache do modelo.

In [9]:
assert digest(MAIN_PATH) == MAIN_SHA256_BEFORE
for filename in ("quality_model.joblib","manifest.json","train_samples.csv","val_samples.csv","test_samples.csv","example_request.json"):
    temporary = VERSION_ROOT / (filename + ".new")
    shutil.copy2(RUN_DIR / filename, temporary)
    temporary.replace(VERSION_ROOT / filename)
assert digest(VERSION_ROOT / "quality_model.joblib") == manifest["model_sha256"]
assert digest(MAIN_PATH) == MAIN_SHA256_BEFORE
print("Versão publicada para o caminho já utilizado pela API:", VERSION_ROOT / "quality_model.joblib")
print("main.py preservado:", MAIN_SHA256_BEFORE)
print("Arquitetura:", manifest["estimator"])

Versão publicada para o caminho já utilizado pela API: C:\Users\lf446\OneDrive\Documentos\GitHub\WaterPath_TCC\back-end\ia\app\services\Predict\model\metals_v2_20261004\quality_model.joblib
main.py preservado: 60c7fab8fc271f4e08a330c5061b97b2d794dcc997527e5582bbf3a5e8cec937
Arquitetura: MultiOutputRegressor(TransformedTargetRegressor(StackingRegressor[RF,XGBoost,CatBoost; RidgeCV], QuantileTransformer))
